# Steam Hit Predictor: Phase 3 - 8-Bit QLoRA Fine-Tuning on Kaggle GPU
Fine-tune **Qwen3.5-0.8B** using **BitsAndBytes 8-bit quantization** and **PEFT LoRA** on Kaggle GPUs to predict game vibe scores (`hype_score`, `story_score`, `nostalgia_score`) from descriptions without data leakage.

### Kaggle Setup Instructions:
1. In the right panel under **Notebook options**, select **Accelerator: GPU** (e.g. GPU T4 x2 or P100).
2. Ensure **Internet: On** is toggled in the right panel.
3. Attach your dataset (e.g. `steam-hit-predictor-cleaned-data` or upload the `cleaned_data` folder).
4. Click **Run All**!


In [ ]:
# 1. Install Dependencies in Kaggle Environment
!pip install -q "transformers>=4.40.0" "peft>=0.10.0" "trl>=0.8.0" "bitsandbytes>=0.43.0" "accelerate>=0.28.0" "datasets>=2.18.0" "tqdm" "kagglehub>=0.4.1"

import os
import sys
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2. Automatically Locate Cleaned Dataset in /kaggle/input/
import os
import pandas as pd

def find_cleaned_data_root(start_dir="/kaggle/input"):
    for root, dirs, files in os.walk(start_dir):
        if "games.csv" in files and os.path.basename(root) == "train":
            return os.path.dirname(root)
        if os.path.basename(root) == "cleaned_data":
            return root
    return "cleaned_data"

DATA_ROOT = find_cleaned_data_root()
print(f"Cleaned dataset path: {DATA_ROOT}")


In [ ]:
# 3. Load & Format Data Splits
def load_split(split="train", max_samples=300):
    g_path = os.path.join(DATA_ROOT, split, "games.csv")
    d_path = os.path.join(DATA_ROOT, split, "games_descriptions.csv")
    df_g = pd.read_csv(g_path)
    df_d = pd.read_csv(d_path)
    df = pd.merge(df_g, df_d, on="appid", how="inner")
    if "clean_description" not in df.columns or df["clean_description"].isna().all():
        df["clean_description"] = df.get("short_description", "")
    df["clean_description"] = df["clean_description"].fillna("").astype(str).str.strip()
    df = df[df["clean_description"].str.len() >= 20].copy()
    for c in ["hype_score", "story_score", "nostalgia_score"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(5.0)
    if max_samples and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42).reset_index(drop=True)
    return df

df_train = load_split("train", max_samples=300)
df_eval = load_split("test", max_samples=20)
print(f"Loaded {len(df_train)} train games and {len(df_eval)} holdout test games.")


In [ ]:
# 4. Initialize Base LLM in 8-Bit Quantization
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto"
)
model.generation_config.top_p = 0.9
model.generation_config.temperature = 0.7
model.generation_config.do_sample = True
print("Base model loaded in 8-bit quantization on GPU!")


In [ ]:
# 5. Configure LoRA Adapter
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
# 6. Build Training Dataset with Vectorized Prompt Mapping
import json
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert game analyst. Analyze the following Steam game description and predict its vibe scores between 1.0 and 10.0:
- hype_score: Anticipation, energy, visual spectacle, and mainstream appeal (1.0 to 10.0).
- story_score: Narrative depth, worldbuilding, emotional engagement, and lore (1.0 to 10.0).
- nostalgia_score: Retro aesthetics, classic mechanics, and vintage resonance (1.0 to 10.0).

Return ONLY valid JSON matching this schema:
{"hype_score": float, "story_score": float, "nostalgia_score": float}"""

def format_prompt(row):
    title = str(row.get("name", "Unknown Game"))
    desc = str(row.get("clean_description", "")).strip()
    return f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nGame Title: {title}\nDescription: {desc}\n\nPredict the vibe scores JSON.<|im_end|>\n<|im_start|>assistant\n"

def format_dataset_batch(batch):
    texts = []
    for i in range(len(batch["name"])):
        target = {
            "hype_score": round(float(batch["hype_score"][i]), 1),
            "story_score": round(float(batch["story_score"][i]), 1),
            "nostalgia_score": round(float(batch["nostalgia_score"][i]), 1)
        }
        prompt = format_prompt({"name": batch["name"][i], "clean_description": batch["clean_description"][i]})
        texts.append(f"{prompt}{json.dumps(target)}<|im_end|>")
    return {"text": texts}

raw_ds = Dataset.from_pandas(df_train[["name", "clean_description", "hype_score", "story_score", "nostalgia_score"]].reset_index(drop=True))
train_ds = raw_ds.map(format_dataset_batch, batched=True, remove_columns=raw_ds.column_names)
print(f"Prepared {len(train_ds)} formatted training samples.")


In [ ]:
# 7. SFTTrainer Fine-Tuning Loop on Kaggle GPU
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/qwen_steam_vibe_8bit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

sft_config = SFTConfig(
    output_dir="/kaggle/working/_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    bf16=False,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
    seed=42
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    args=sft_config,
    processing_class=tokenizer,
)

print("Executing 8-bit QLoRA fine-tuning loop...")
trainer.train()

# Save adapter to /kaggle/working/ (persisted by Kaggle as output artifact)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Successfully saved fine-tuned adapter artifacts to: {OUTPUT_DIR}")


In [ ]:
# 8. Evaluation Benchmark on Holdout Test Set
import re

model.eval()
tokenizer.padding_side = "left"
total_samples = len(df_eval)
valid_count = 0
errors = []
batch_size = 4

for i in range(0, total_samples, batch_size):
    batch = df_eval.iloc[i : min(i + batch_size, total_samples)]
    prompts = [format_prompt(row) for _, row in batch.iterrows()]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            top_p=0.9,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    for idx, (out_ids, in_ids) in enumerate(zip(outputs, inputs.input_ids)):
        gen_text = tokenizer.decode(out_ids[len(in_ids):], skip_special_tokens=True).strip()
        row = batch.iloc[idx]
        match = re.search(r"\{[^{}]*\}", gen_text)
        if match:
            try:
                data = json.loads(match.group(0))
                errors.extend([
                    abs(float(data["hype_score"]) - float(row["hype_score"])),
                    abs(float(data["story_score"]) - float(row["story_score"])),
                    abs(float(data["nostalgia_score"]) - float(row["nostalgia_score"]))
                ])
                valid_count += 1
            except Exception:
                pass

mae = sum(errors) / max(len(errors), 1)
acc = sum(1 for e in errors if e <= 2.0) / max(len(errors), 1) * 100.0
print(f"Holdout Schema Validity: {valid_count}/{total_samples} ({valid_count/total_samples*100:.1f}%)")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Accuracy within +-2.0:     {acc:.1f}%")


In [ ]:
# 9. List Artifacts Saved in Kaggle Working Directory
!ls -lh /kaggle/working/qwen_steam_vibe_8bit/
